# ColabのL4 GPUでOllamaを立てる

ローカルの `local_llm` RAGアプリから、このColabノートブック上のOllama（L4 GPU）にAPI経由で接続するためのセットアップ。

**手順**
1. ランタイム → ランタイムのタイプを変更 → GPU（L4）を選択してから、上から順にセルを実行する
2. ngrokの無料アカウントを作り、ダッシュボードでauthtokenを控えておく（未取得なら ngrok.com で登録できる）
3. 最後のセルで表示される `OLLAMA_HOST` と `OLLAMA_API_KEY` を、ローカル側の `.env`（`.env.example` をコピーして作る）に設定する
4. ローカルで `myvenv313\Scripts\python.exe -m streamlit run rag_chat_app.py` を起動する

**注意**
- Colabのランタイムはアイドルや時間経過で切断される。常時稼働のAPIではなく、使う時だけ起動するものとして扱うこと。
- ここで発行されるngrokのURLは誰でも到達できるが、`X-API-Key` ヘッダーが一致しないリクエストはプロキシが401で拒否する。それでもURLとAPIキーは他人に共有しないこと。

In [ ]:
# 1. Ollama本体のインストール
# Ollamaのインストーラは展開にzstdを、GPU検出にlspci/lshwを使う。
# どちらもColabランタイムに入っていないため事前に入れておく（lspci/lshwが無いと
# 「Unable to detect NVIDIA/AMD GPU」の警告が出てGPU依存関係のセットアップが
# 丸ごとスキップされ、L4を使わずCPUで動いてしまう）
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Ollamaサーバーをバックグラウンドで起動し、応答を待つ
import os
import subprocess
import time

import requests

# gpt-oss:20b のコーディング用途では64kを既定にする。
# L4のVRAMに収まらない場合は、32768に変更してからこのセルを再実行する。
OLLAMA_CONTEXT_LENGTH = 65536
ollama_env = os.environ.copy()
ollama_env["OLLAMA_CONTEXT_LENGTH"] = str(OLLAMA_CONTEXT_LENGTH)
ollama_env["OLLAMA_NUM_PARALLEL"] = "1"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=ollama_env,
)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama起動確認OK")
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollamaの起動に失敗しました")

# L4 GPUをOllamaが認識できていることを確認する。ここで失敗する場合はCPUで動いて
# しまうため、ランタイムがGPU（L4）かを確認してセル1からやり直す。
gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError(
        "GPUが見えていません。ランタイムのタイプがGPU（L4）か確認してください: "
        + gpu_check.stderr
    )
print("GPU確認OK:", gpu_check.stdout.strip())


In [ ]:
# 3. モデルを取得する（RAG用 + VLM用 + コーディング用。時間がかかる）
!ollama pull bge-m3
!ollama pull qwen2.5vl:7b
!ollama pull gpt-oss:20b


In [ ]:
# 4. 認証つきリバースプロキシに必要なパッケージ
!pip install -q fastapi "uvicorn[standard]" httpx pyngrok

In [ ]:
# 5. プロキシ用のAPIキーを発行する。この値をローカルの .env の OLLAMA_API_KEY に設定する
import os
import secrets

OLLAMA_PROXY_API_KEY = secrets.token_urlsafe(24)
# プロキシは起動時にこの環境変数を読む。生成ファイルへキーを埋め込まない。
os.environ["OLLAMA_PROXY_API_KEY"] = OLLAMA_PROXY_API_KEY
print("ローカル .env に設定する OLLAMA_API_KEY:")
print(OLLAMA_PROXY_API_KEY)


In [ ]:
# 6. リバースプロキシ本体を書き出す。
# X-API-Keyヘッダーを検証したうえで、パスをそのまま127.0.0.1:11434のOllamaへ
# ストリーミング転送する（RAGとResponses APIのSSEストリームもそのまま通す）。
from pathlib import Path

proxy_code = """
import os

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

API_KEY = os.environ["OLLAMA_PROXY_API_KEY"]
OLLAMA_BASE = "http://127.0.0.1:11434"

app = FastAPI()
_client = httpx.AsyncClient(
    base_url=OLLAMA_BASE,
    timeout=httpx.Timeout(connect=10.0, read=600.0, write=60.0, pool=10.0),
)

_HOP_BY_HOP = {"host", "x-api-key", "authorization", "content-length", "connection", "transfer-encoding"}


@app.api_route("/{path:path}", methods=["GET", "POST"])
async def proxy(path: str, request: Request):
    if request.headers.get("x-api-key") != API_KEY:
        return JSONResponse({"error": "unauthorized"}, status_code=401)

    body = await request.body()
    upstream_req = _client.build_request(
        request.method,
        f"/{path}",
        params=request.query_params,
        content=body,
        headers=[(k, v) for k, v in request.headers.items() if k.lower() not in _HOP_BY_HOP],
    )
    try:
        upstream = await _client.send(upstream_req, stream=True)
    except httpx.TimeoutException:
        return JSONResponse(
            {"error": "Ollama upstream timed out"}, status_code=504
        )
    except httpx.RequestError:
        return JSONResponse(
            {"error": "Unable to reach Ollama upstream"}, status_code=502
        )

    async def body_iter():
        try:
            async for chunk in upstream.aiter_raw():
                yield chunk
        finally:
            await upstream.aclose()

    return StreamingResponse(
        body_iter(),
        status_code=upstream.status_code,
        headers={k: v for k, v in upstream.headers.items() if k.lower() not in _HOP_BY_HOP},
    )
"""

Path("ollama_proxy.py").write_text(proxy_code, encoding="utf-8")
print("ollama_proxy.py を書き出しました")


In [ ]:
# 7. プロキシをバックグラウンドで起動し、応答を待つ
# セル5(APIキー再発行)やこのセル自体を再実行すると、直前に起動したプロキシが
# ポート8000を掴んだまま古いAPIキーで生き続け、新しいプロキシがバインドに失敗して
# 即終了する（ヘルスチェックは古いプロセスに当たり401が返り続けて原因不明のまま
# タイムアウトする）。再実行しても安全なように、起動前に前回のプロセスを必ず止める。
if "proxy_proc" in globals() and proxy_proc.poll() is None:
    proxy_proc.terminate()
    proxy_proc.wait(timeout=10)

proxy_log_path = Path("proxy.log")
proxy_log = proxy_log_path.open("w", encoding="utf-8")
proxy_proc = subprocess.Popen(
    ["uvicorn", "ollama_proxy:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=proxy_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        r = requests.get(
            "http://127.0.0.1:8000/api/tags",
            headers={"X-API-Key": OLLAMA_PROXY_API_KEY},
            timeout=2,
        )
        if r.status_code == 200:
            print("プロキシ起動確認OK")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    proxy_log.flush()
    print("--- proxy.log ---")
    print(proxy_log_path.read_text(encoding="utf-8", errors="replace"))
    raise RuntimeError("プロキシの起動に失敗しました")

In [ ]:
# 8. ngrokでプロキシ（8000番）を外部公開する。
# authtokenはngrokダッシュボード（https://dashboard.ngrok.com/get-started/your-authtoken）から取得する
from getpass import getpass

from pyngrok import conf, ngrok

conf.get_default().auth_token = getpass("ngrok authtoken: ")
public_url = ngrok.connect(8000, "http")
print("ローカル .env に設定する OLLAMA_HOST:")
print(public_url)

In [ ]:
# GPU使用量を確認する。pullだけではモデルはロードされないため、RAGまたはCodexから
# gpt-oss:20bへ一度リクエストを送ってから実行する。
!ollama ps
!nvidia-smi

print(proxy_log_path.read_text(encoding="utf-8", errors="replace")[-3000:])


## ローカル側の設定

`local_llm/.env.example` を `.env` にコピーし、上のセルで表示された値を設定する。

```
OLLAMA_HOST=<上で表示された public_url>
OLLAMA_API_KEY=<セル5で表示された OLLAMA_PROXY_API_KEY>
```

保存したら、いつも通りローカルで起動する。

```powershell
myvenv313\Scripts\python.exe -m streamlit run rag_chat_app.py
```

RAGアプリでは埋め込みに `bge-m3`、画像処理に `qwen2.5vl:7b` を使う。コーディングエージェントは `gpt-oss:20b` を使う。

In [ ]:
# 9. 使い終わったら実行して後片付けする（トンネル・プロキシ・Ollamaを止める）
# ngrok.disconnect(public_url.public_url)
# proxy_proc.terminate()
# ollama_proc.terminate()
# print("停止しました")